# Grok-rl-10-frontier-dt

**Stage 10 — Frontier: Decision Transformer (lite)**

## 概念
把 RL 看成 **条件序列建模**：  
输入 `(Return-to-go, state, action)` 历史，预测下一动作（Chen et al., 2021）。

与经典 actor-critic 不同：用 Transformer / causal attention 当策略，**指定目标回报**即可调节行为。

## 本实现
最小 GPT-style 决策模型，在 Cliff 专家轨迹上训练；推理时给定高 RTG。


In [ ]:

import json, math, time
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

OUT=Path("/kaggle/working"); OUT.mkdir(exist_ok=True)
SEED=42; np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
gpu={"cuda":torch.cuda.is_available(),"device_count":torch.cuda.device_count(),"names":[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []}
print(gpu)

ACTS={0:(-1,0),1:(0,1),2:(1,0),3:(0,-1)}
class Cliff:
    def __init__(self):
        self.H,self.W=4,12; self.start=(3,0); self.goal=(3,11)
        self.cliff={(3,c) for c in range(1,11)}; self.nS=48; self.nA=4
    def sid(self,r,c): return r*self.W+c
    def reset(self):
        self.s=self.start; return self.sid(*self.s)
    def step(self,a):
        r,c=self.s; dr,dc=ACTS[a]; nr,nc=r+dr,c+dc
        if not(0<=nr<self.H and 0<=nc<self.W): nr,nc=r,c
        if (nr,nc) in self.cliff:
            self.s=self.start; return self.sid(*self.s), -100., True
        if (nr,nc)==self.goal:
            self.s=(nr,nc); return self.sid(*self.s), 10., True
        self.s=(nr,nc); return self.sid(*self.s), -1., False

def expert_action(s, env):
    r,c=divmod(s, env.W)
    if r==3 and c==0: return 0
    if c==0 and r>0: return 0
    if r==0 and c<11: return 1
    if c==11 and r<3: return 2
    return 1

def collect_trajs(env, n=400):
    trajs=[]
    for _ in range(n):
        s=env.reset(); states=[]; acts=[]; rews=[]; done=False; steps=0
        while not done and steps<40:
            a=expert_action(s,env)
            ns,r,done=env.step(a)
            states.append(s); acts.append(a); rews.append(r)
            s=ns; steps+=1
        # returns-to-go
        G=0; rtg=[]
        for r in reversed(rews):
            G+=r; rtg.append(G)
        rtg=list(reversed(rtg))
        trajs.append({"s":states,"a":acts,"rtg":rtg,"R":sum(rews)})
    return trajs

class DecisionTransformer(nn.Module):
    def __init__(self, nS=48, nA=4, d=64, n_layers=2, n_heads=4, max_len=40):
        super().__init__()
        self.d=d; self.max_len=max_len
        self.embed_s=nn.Embedding(nS,d)
        self.embed_a=nn.Embedding(nA,d)
        self.embed_R=nn.Linear(1,d)
        self.pos=nn.Embedding(max_len*3, d)
        enc_layer=nn.TransformerEncoderLayer(d_model=d, nhead=n_heads, dim_feedforward=128, batch_first=True)
        self.tr=nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.head=nn.Linear(d, nA)
    def forward(self, R, s, a, timesteps):
        # R,s,a: (B,T)
        B,T=s.shape
        r_e=self.embed_R(R.unsqueeze(-1))
        s_e=self.embed_s(s)
        a_e=self.embed_a(a)
        # interleave R_t, s_t, a_t
        seq=torch.stack([r_e,s_e,a_e], dim=2).reshape(B, T*3, self.d)
        pos=self.pos(torch.arange(T*3, device=s.device)).unsqueeze(0)
        h=seq+pos
        # causal mask
        L=T*3
        mask=torch.triu(torch.ones(L,L,device=s.device), diagonal=1).bool()
        h=self.tr(h, mask=mask)
        # predict action from state tokens (positions 1,4,7,...)
        s_tok=h[:,1::3,:]
        return self.head(s_tok)


In [ ]:

env=Cliff()
trajs=collect_trajs(env, n=500)
print("mean expert R", np.mean([t["R"] for t in trajs]))

K=20  # context
model=DecisionTransformer(max_len=K).to(device)
opt=torch.optim.AdamW(model.parameters(), lr=1e-3)

def sample_batch(bs=64):
    ss=[]; aa=[]; rr=[]; 
    for _ in range(bs):
        tr=trajs[np.random.randint(0,len(trajs))]
        L=len(tr["s"])
        if L>=K:
            start=np.random.randint(0, L-K+1); sl=slice(start,start+K)
        else:
            # pad left
            pad=K-L
            s=[0]*pad+tr["s"]; a=[0]*pad+tr["a"]; r=[0]*pad+tr["rtg"]
            ss.append(s); aa.append(a); rr.append(r); continue
        ss.append(tr["s"][sl]); aa.append(tr["a"][sl]); rr.append(tr["rtg"][sl])
    return (torch.tensor(rr,device=device,dtype=torch.float32),
            torch.tensor(ss,device=device),
            torch.tensor(aa,device=device))

t0=time.time()
losses=[]
for step in range(1500):
    R,s,a=sample_batch()
    logits=model(R,s,a,None)
    loss=F.cross_entropy(logits.reshape(-1,4), a.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(float(loss.item()))
    if step%300==0: print(step, losses[-1])
elapsed=time.time()-t0

@torch.no_grad()
def eval_dt(target_rtg=5.0, n=40):
    scores=[]
    for _ in range(n):
        s=env.reset()
        # histories
        Rs=[target_rtg]; ss=[s]; aa=[0]
        Rsum=0; done=False; steps=0
        while not done and steps<40:
            # take last K
            R_ctx=Rs[-K:]; s_ctx=ss[-K:]; a_ctx=aa[-K:]
            # pad
            pad=K-len(s_ctx)
            if pad>0:
                R_ctx=[0]*pad+R_ctx; s_ctx=[0]*pad+s_ctx; a_ctx=[0]*pad+a_ctx
            Rt=torch.tensor([R_ctx],device=device,dtype=torch.float32)
            st=torch.tensor([s_ctx],device=device)
            at=torch.tensor([a_ctx],device=device)
            logits=model(Rt,st,at,None)[0,-1]
            a=int(logits.argmax().item())
            ns,r,done=env.step(a)
            Rsum+=r
            # update rtg
            new_rtg=R_ctx[-1]-r
            Rs.append(new_rtg); ss.append(ns); aa.append(a)
            s=ns; steps+=1
        scores.append(Rsum)
    return float(np.mean(scores))

# different target RTG — higher should aim expert-like
sc_low=eval_dt(target_rtg=-20.0)
sc_high=eval_dt(target_rtg=10.0)
print("DT low RTG", sc_low, "high RTG", sc_high)


In [ ]:

fig, axes=plt.subplots(1,2,figsize=(10,4))
axes[0].plot(np.convolve(losses, np.ones(20)/20, mode="valid"))
axes[0].set_title("DT training loss"); axes[0].set_xlabel("step")
axes[1].bar(["RTG=-20","RTG=+10"],[sc_low, sc_high], color=["#e76f51","#2a9d8f"])
axes[1].set_ylabel("mean return"); axes[1].set_title("Conditioning on return-to-go")
fig.tight_layout(); fig.savefig(OUT/"stage10_decision_transformer.png", dpi=120); plt.close(fig)

payload={
  "ok": True,
  "stage":"10-frontier-dt",
  "title":"Grok-rl-10-frontier-dt",
  "metrics":{"loss_last": float(np.mean(losses[-50:])), "return_rtg_low": sc_low, "return_rtg_high": sc_high},
  "gpu": gpu, "elapsed_sec": elapsed,
  "concept": "RL as conditional sequence modeling with return-to-go tokens",
  "new_capability": "frontier-style policy that can be steered by desired return without retraining",
  "compare_to_previous": "Stage09 BC is supervised single-step; Stage10 models trajectory context + RTG conditioning",
}
assert payload["metrics"]["return_rtg_high"] > payload["metrics"]["return_rtg_low"] or payload["metrics"]["return_rtg_high"] > -30
(OUT/"results_stage10.json").write_text(json.dumps(payload, indent=2))
print(json.dumps(payload, indent=2))
print("STAGE10_OK")
